# 02. Limpieza y preparación de datos - Superstore

**Objetivo:** dejar los datos listos para el modelamiento, tomando decisiones justificadas y evitando fuga de datos.

Pasos: conversión de fechas, duplicados, variables derivadas, tratamiento de atípicos, eliminación de columnas, separación en train y test, codificación de categóricas y exportación.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.carga_datos import cargar_datos, guardar_csv
from src.limpieza import convertir_fechas, eliminar_duplicados, eliminar_columnas
from src.transformacion import crear_variables_derivadas, separar_train_test, codificar_categoricas

## 1. Carga de los datos originales
Se carga el archivo de `data/raw`, que no se modifica nunca. Todo lo que se transforma se guarda en otras carpetas.

In [2]:
df = cargar_datos("../data/raw/Sample_-_Superstore.csv")
print(df.shape)

(9994, 21)


## 2. Conversión de fechas
Las fechas vienen como texto con formato mes/día/año. Se convierten a tipo fecha para poder restarlas y extraer el año, el mes y el día de la semana.

In [3]:
df = convertir_fechas(df, ["Order Date", "Ship Date"])
print(df[["Order Date", "Ship Date"]].dtypes)
print("Rango de fechas de pedido:", df["Order Date"].min().date(), "a", df["Order Date"].max().date())

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object
Rango de fechas de pedido: 2014-01-03 a 2017-12-30


## 3. Duplicados
En la exploración no se encontraron duplicados. Se aplica de todas formas la función para dejar el proceso completo y reproducible.

In [4]:
df = eliminar_duplicados(df)
print(df.shape)

Duplicados encontrados: 0
(9994, 21)


## 4. Variables derivadas

Se crean variables simples:

- `Anio_Pedido`, `Mes_Pedido`, `Dia_Semana_Pedido`: para ver si hay un patrón por época.
- `Dias_Envio`: días entre el pedido y el envío.
- `Sales_Log`: logaritmo de `Sales` (ver sección de atípicos).

**No** se crea ninguna variable a partir de `Profit`, porque eso le daría al modelo parte de la respuesta (fuga de datos).

In [5]:
df = crear_variables_derivadas(df)
df[["Order Date", "Ship Date", "Anio_Pedido", "Mes_Pedido", "Dia_Semana_Pedido", "Dias_Envio", "Sales", "Sales_Log"]].head()

,Order Date,Ship Date,Anio_Pedido,Mes_Pedido,Dia_Semana_Pedido,Dias_Envio,Sales,Sales_Log
0,2016-11-08,2016-11-11,2016,11,1,3,261.9600,5.572002
1,2016-11-08,2016-11-11,2016,11,1,3,731.9400,6.597064
2,2016-06-12,2016-06-16,2016,6,6,4,14.6200,2.748552
3,2015-10-11,2015-10-18,2015,10,6,7,957.5775,6.865450
4,2015-10-11,2015-10-18,2015,10,6,7,22.3680,3.151368


## 5. Tratamiento de valores atípicos

En la exploración vimos que `Profit` y `Sales` tienen muchos valores extremos. Decisiones:

- **`Profit` (variable objetivo): no se elimina ni se recorta.** Los extremos son transacciones reales (por ejemplo, ventas grandes de copiers o pérdidas por descuentos altos). Si se eliminan, el modelo no aprendería justamente los casos más importantes para el negocio.
- **`Sales`: se agrega una versión logarítmica (`Sales_Log`).** La transformación no necesita aprender nada de los datos, así que se puede aplicar antes de separar train y test sin fuga de datos. Se conserva también `Sales` original.

In [6]:
print("Asimetría de Sales:     ", round(df["Sales"].skew(), 2))
print("Asimetría de Sales_Log: ", round(df["Sales_Log"].skew(), 2))
print("Asimetría de Profit (se deja igual):", round(df["Profit"].skew(), 2))

Asimetría de Sales:      12.97
Asimetría de Sales_Log:  0.28
Asimetría de Profit (se deja igual): 7.56


## 6. Eliminación de columnas irrelevantes

| Columna | Motivo |
|---|---|
| `Row ID` | Es solo un consecutivo. |
| `Country` | Tiene un único valor. |
| `Customer ID`, `Customer Name` | Más de 790 valores distintos; identifican a la persona, no explican la ganancia. |
| `Product ID`, `Product Name` | Más de 1.800 valores distintos; la información de producto ya está en `Category` y `Sub-Category`. |
| `City`, `State`, `Postal Code` | Demasiados valores distintos; se deja `Region` como variable geográfica. |
| `Order Date`, `Ship Date` | Ya se extrajo su información en las variables derivadas. |

`Order ID` se conserva por ahora porque se necesita para separar train y test sin mezclar filas de un mismo pedido. Se elimina después de la separación.

In [7]:
columnas_a_eliminar = [
    "Row ID", "Country", "Customer ID", "Customer Name",
    "Product ID", "Product Name", "City", "State", "Postal Code",
    "Order Date", "Ship Date",
]
df_limpio = eliminar_columnas(df, columnas_a_eliminar)
print(df_limpio.shape)
df_limpio.head()

(9994, 15)


,Order ID,Ship Mode,Segment,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Anio_Pedido,Mes_Pedido,Dia_Semana_Pedido,Dias_Envio,Sales_Log
0,CA-2016-152156,Second Class,Consumer,South,Furniture,Bookcases,261.9600,2,0.00,41.9136,2016,11,1,3,5.572002
1,CA-2016-152156,Second Class,Consumer,South,Furniture,Chairs,731.9400,3,0.00,219.5820,2016,11,1,3,6.597064
2,CA-2016-138688,Second Class,Corporate,West,Office Supplies,Labels,14.6200,2,0.00,6.8714,2016,6,6,4,2.748552
3,US-2015-108966,Standard Class,Consumer,South,Furniture,Tables,957.5775,5,0.45,-383.0310,2015,10,6,7,6.865450
4,US-2015-108966,Standard Class,Consumer,South,Office Supplies,Storage,22.3680,2,0.20,2.5164,2015,10,6,7,3.151368


Se guarda el dataset limpio (aún sin codificar) en `data/modificado`.

In [8]:
guardar_csv(df_limpio, "../data/modificado/superstore_limpio.csv")

Archivo guardado en ../data/modificado/superstore_limpio.csv con forma (9994, 15)


## 7. Separación en train y test

Se separa **antes** de codificar para que el conjunto de prueba no influya en ninguna decisión. Se usa 80% para entrenamiento y 20% para prueba, con `random_state=42`.

Como un pedido tiene varias filas, la separación se hace por `Order ID`: todas las filas de un pedido quedan juntas en el mismo conjunto. Si no se hiciera así, el modelo podría ver en train filas casi iguales a las de test y las métricas saldrían más optimistas de lo real.

In [9]:
train, test = separar_train_test(df_limpio, columna_grupo="Order ID", tamanio_test=0.2)

print("Filas en train:", len(train), f"({len(train) / len(df_limpio):.1%})")
print("Filas en test: ", len(test), f"({len(test) / len(df_limpio):.1%})")

pedidos_compartidos = set(train["Order ID"]) & set(test["Order ID"])
print("Pedidos que están en ambos conjuntos:", len(pedidos_compartidos))

Filas en train: 7959 (79.6%)
Filas en test:  2035 (20.4%)
Pedidos que están en ambos conjuntos: 0


In [10]:
train = train.drop(columns=["Order ID"])
test = test.drop(columns=["Order ID"])

## 8. Codificación de variables categóricas

Las variables categóricas (`Ship Mode`, `Segment`, `Region`, `Category`, `Sub-Category`) no tienen un orden natural, por eso se usa One-Hot Encoding en lugar de asignarles números 1, 2, 3 (que le sugerirían al modelo un orden que no existe).

Las columnas se definen con **train** y `test` se ajusta a esas mismas columnas.

In [11]:
columnas_categoricas = ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]
train_cod, test_cod = codificar_categoricas(train, test, columnas_categoricas)

print("Columnas train:", train_cod.shape[1])
print("Columnas test: ", test_cod.shape[1])
print("¿Mismas columnas y en el mismo orden?", list(train_cod.columns) == list(test_cod.columns))

Columnas train: 40
Columnas test:  40
¿Mismas columnas y en el mismo orden? True


In [12]:
train_cod.head()

,Sales,Quantity,Discount,Profit,Anio_Pedido,Mes_Pedido,Dia_Semana_Pedido,Dias_Envio,Sales_Log,Ship Mode_First Class,...,Sub-Category_Envelopes,Sub-Category_Fasteners,Sub-Category_Furnishings,Sub-Category_Labels,Sub-Category_Machines,Sub-Category_Paper,Sub-Category_Phones,Sub-Category_Storage,Sub-Category_Supplies,Sub-Category_Tables
0,261.9600,2,0.00,41.9136,2016,11,1,3,5.572002,0,...,0,0,0,0,0,0,0,0,0,0
1,731.9400,3,0.00,219.5820,2016,11,1,3,6.597064,0,...,0,0,0,0,0,0,0,0,0,0
2,14.6200,2,0.00,6.8714,2016,6,6,4,2.748552,0,...,0,0,0,1,0,0,0,0,0,0
3,957.5775,5,0.45,-383.0310,2015,10,6,7,6.865450,0,...,0,0,0,0,0,0,0,0,0,1
4,22.3680,2,0.20,2.5164,2015,10,6,7,3.151368,0,...,0,0,0,0,0,0,0,1,0,0


## 9. Exportación de los datos procesados

In [13]:
guardar_csv(train_cod, "../data/train/train.csv")
guardar_csv(test_cod, "../data/test/test.csv")

Archivo guardado en ../data/train/train.csv con forma (7959, 40)
Archivo guardado en ../data/test/test.csv con forma (2035, 40)


In [14]:
print("Nulos en train:", train_cod.isnull().sum().sum())
print("Nulos en test: ", test_cod.isnull().sum().sum())
print("Profit promedio en train:", round(train_cod["Profit"].mean(), 2))
print("Profit promedio en test: ", round(test_cod["Profit"].mean(), 2))

Nulos en train: 0
Nulos en test:  0
Profit promedio en train: 27.03
Profit promedio en test:  35.02


## 10. Conclusiones de la limpieza

Partí de 9.994 registros y 21 columnas. Después de eliminar columnas y quitar `Order ID` quedaron 14 columnas (incluyendo `Profit`), y tras la codificación quedaron 40. No se perdió ninguna fila porque no había nulos ni duplicados.

**Transformaciones principales.** Convertí las fechas y creé cuatro variables a partir de ellas (año, mes, día de la semana y días de envío). También creé `Sales_Log`, que bajó la asimetría de `Sales` de 12,97 a 0,28, es decir, la distribución quedó mucho más simétrica sin eliminar ningún dato.

**Valores atípicos.** Decidí no tocar los valores extremos de `Profit`. Son ventas y pérdidas reales y el modelo debe intentar predecirlas. La consecuencia es que las métricas de error, sobre todo el RMSE, van a ser altas porque unos pocos registros grandes pesan mucho; esto se tiene que tener en cuenta al interpretar los modelos.

**Separación y fuga de datos.** Separé los datos antes de codificar y por pedido (`Order ID`). Comprobé que ningún pedido quedó en los dos conjuntos. La proporción quedó en 79,6% y 20,4% porque se agrupa por pedido y no por fila. El Profit promedio es distinto entre train (27,03) y test (35,02); es normal porque el Profit tiene mucha varianza, pero explica que los resultados dependan un poco de cómo cayeron los registros extremos.

**Limitaciones.** Al eliminar `State`, `City` y los datos de cliente y producto se pierde información que podría ayudar al modelo. Lo hice para mantener el proyecto sencillo; en una versión futura se podría probar agrupar estos datos de otra manera.